Python notebook to convert gliner saved results to Label studio annotations.

## Data and Libraries

In [1]:
import json
import uuid
import os
import re

In [2]:
def extract_paper_id(filename):
    match = re.match(r"\d+_(\d+)_", filename)
    return match.group(1) if match else "unknown"

def convert_to_labelstudio_format(data, paper_id, model_version="gliner-community/gliner_medium-v2.5"):
    if not isinstance(data, list):
        raise TypeError("Expected data to be a list of dictionaries, but got: {}".format(type(data)))
    
    labelstudio_data = []
    
    for item in data:
        if not isinstance(item, dict) or "entities" not in item:
            continue
        
        predictions = {
            "model_version": model_version,
            "score": 0.5,  # Default score for the overall prediction
            "result": []
        }
        
        for entity in item.get("entities", []):
            if not isinstance(entity, dict):
                continue
            
            predictions["result"].append({
                "id": str(uuid.uuid4())[:8],
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": entity.get("start", 0),
                    "end": entity.get("end", 0),
                    "score": entity.get("score", 0.0),
                    "text": entity.get("text", ""),
                    "labels": [entity.get("label", "Unknown")]
                }
            })
        
        item["paper_id"] = paper_id
        labelstudio_data.append({"data": item, "predictions": [predictions]})
    
    return labelstudio_data

def process_directory(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            paper_id = extract_paper_id(filename)
            
            with open(input_path, "r", encoding="utf-8") as infile:
                try:
                    data = json.load(infile)
                except json.JSONDecodeError:
                    print(f"Skipping {filename}: Invalid JSON format.")
                    continue
            
            try:
                transformed_data = convert_to_labelstudio_format(data, paper_id)
            except TypeError as e:
                print(f"Skipping {filename}: {e}")
                continue
            
            with open(output_path, "w", encoding="utf-8") as outfile:
                json.dump(transformed_data, outfile, indent=4)

In [3]:
# Example usage (Change accordingly)
input_directory = "RESULTS/GLINER_MEDIUM_V25_V1_PSS/"
output_directory = "RESULTS/GLINER_MEDIUM_V25_V1_PSS_LS/"
process_directory(input_directory, output_directory)
